# Process an image

## References

- Harpy: Rombaut, B. et al. Scalable analysis of whole slide spatial proteomics with Harpy. Bioinformatics btag122 (2026) doi:10.1093/bioinformatics/btag122.
- Spatialdata: Marconato, L. et al. SpatialData: an open and universal data framework for spatial omics. Nat Methods 1–5 (2024) doi:10.1038/s41592-024-02212-x.
- pyLMD: Schmacke, N. A. et al. SPARCS, a platform for genome-scale CRISPR screening for spatial cellular phenotypes. Preprint at https://doi.org/10.1101/2023.06.01.542416 (2023).



## Import

In [ ]:
from pathlib import Path
import harpy
import lmd.lib as pylmd
import spatialdata as sd
from spatialdata_io.readers.generic import image as spatialdata_imread
from napari_spatialdata import Interactive
from dask.distributed import Client
import spatialdata_plot  # noqa
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Utility functions
from src.utils import random_sample_by_area

/Users/lucas-diedrich/mamba/envs/npdvp2/lib/python3.13/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [ ]:
image_path = Path(
    "../../data/2025_02_19_wholeTissue_melanomaAb_01-Scene-2-ScanRegion1big_c1-3.tiff"
)
results_path = Path("../../results")

In [ ]:
SUBSET_SDATA = True

## Create spatialdata 

In [ ]:
image = spatialdata_imread(
    image_path, data_axes=("x", "y", "c"), coordinate_system="global"
)
sdata = sd.SpatialData(images={"image": image})

In [ ]:
# Write to disk to execute task graph (speeds up downstream workflow if data is loaded lazily)
sdata.write(
    results_path
    / "2025_02_19_wholeTissue_melanomaAb_01-Scene-2-ScanRegion1big_c1-3.sdata.zarr"
)

if SUBSET_SDATA:
    subset = sdata.query.bounding_box(
        axes=("y", "x"),
        min_coordinate=[40000, 5000],
        max_coordinate=[55000, 20000],
        target_coordinate_system="global",
    )
    subset.write(results_path / "subset.sdata.zarr")

In [ ]:
if SUBSET_SDATA:
    sdata = sd.read_zarr(results_path / "subset.sdata.zarr")
else:
    sdata = sd.read_zarr(
        results_path
        / "2025_02_19_wholeTissue_melanomaAb_01-Scene-2-ScanRegion1big_c1-3.sdata.zarr"
    )

sdata.pl.render_images().pl.show()

## Segment
**Critical step**

In [ ]:
# Runs 22 Minutes 55 Seconds on a MacBook Pro (M3)
client = Client()
print(f"Open dashboard at {client.dashboard_link}")


expected_diameter = 50
cellpose_kwargs = {
    # Additional keyword arguments passed to the provided model
    "channels": None,  # cellpose 4 uses the first 3 channels
    "normalize": True,
    "diameter": expected_diameter,  # Diameter is an important parameter for cellpose 3
    "flow_threshold": 0.4,
    "cellprob_threshold": 0,
    "pretrained_model": "cpsam",
}

# Perform segmentation with a vanilla cellpose model
_ = harpy.im.segment(
    sdata=sdata,
    img_layer="image",
    model=harpy.im.cellpose_callable,
    output_labels_layer="segmentation_mask",
    output_shapes_layer="segmentation_boundaries",
    **cellpose_kwargs,
    device=None,  # autodetect device
)

client.shutdown()

**Evaluate**

In [ ]:
# Interactively
session = Interactive(sdata)
session.run()

# Static
sdata.pl.render_images("image").pl.render_labels("segmentation_mask").pl.show()

### Extract features

In [ ]:
_ = harpy.tb.allocate_intensity(
    sdata,
    img_layer="image",
    labels_layer="segmentation_mask",
    output_layer="segmentation_features",
    mode="sum",
)
_ = harpy.tb.add_regionprops(
    sdata,
    labels_layer="segmentation_mask",
    table_layer="segmentation_features",
    output_layer="segmentation_features",
    overwrite=True,
)

### Preprocess

In [ ]:
channels = sdata["segmentation_features"].var_names

# Assign to adata.obs for simpler interaction
sdata["segmentation_features"].obs[[f"{channel}_intensity" for channel in channels]] = (
    sdata["segmentation_features"].X
)

for channel in channels:
    sdata["segmentation_features"].obs[f"{channel}_intensity__area_normalized"] = (
        sdata["segmentation_features"].obs[f"{channel}_intensity"]
        / sdata["segmentation_features"].obs["area"]
    )

### Visualize

## QC 

In [ ]:
features = [
    "area",
    "0_intensity__area_normalized",
    "1_intensity__area_normalized",
    "2_intensity__area_normalized",
]

fig, axs = plt.subplots(1, len(features), figsize=(3 * len(features), 3), squeeze=False)
for feature, ax in zip(features, axs.ravel(), strict=True):
    sns.histplot(data=sdata["segmentation_features"].obs, x=feature, ax=ax)
    ax.set_title(f"Channel: {feature}", loc="left", fontsize=8)
    ax.set_xlabel("Area Normalized Intensity")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
area_outlier = (sdata["segmentation_features"].obs["area"] < 100) | (
    sdata["segmentation_features"].obs["area"] > 30000
)


cytosol_channel_intensities = sdata["segmentation_features"].obs[
    "1_intensity__area_normalized"
]
cytosol_channel_outlier = (
    cytosol_channel_intensities < np.quantile(cytosol_channel_intensities, 0.01)
) | (cytosol_channel_intensities > np.quantile(cytosol_channel_intensities, 0.99))

nucleus_channel_intensities = sdata["segmentation_features"].obs[
    "2_intensity__area_normalized"
]
nucleus_channel_outlier = (
    nucleus_channel_intensities < np.quantile(nucleus_channel_intensities, 0.01)
) | (nucleus_channel_intensities > np.quantile(nucleus_channel_intensities, 0.99))

marker_channel_intensities = sdata["segmentation_features"].obs[
    "0_intensity__area_normalized"
]
marker_channel_outlier = marker_channel_intensities > np.quantile(
    marker_channel_intensities, 0.99
)

sdata["segmentation_features"].obs["segmentation_outlier"] = (
    area_outlier
    | cytosol_channel_outlier
    | nucleus_channel_outlier
    | marker_channel_outlier
)

In [ ]:
(
    sdata
    # .pl.render_images("image")
    .pl.render_labels(
        "segmentation_mask", color="0_intensity__area_normalized"
    ).pl.show(title="Area normalized intensity (Sox10)")
)

(
    sdata
    # .pl.render_images("image")
    .pl.render_labels("segmentation_mask", color="segmentation_outlier").pl.show(
        title="Segmentation outlier"
    )
)

## Classification

### Threshold

In [ ]:
marker_intensity_cutoff = 100

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
(
    sdata["segmentation_features"]
    .obs["0_intensity__area_normalized"]
    .loc[~sdata["segmentation_features"].obs["segmentation_outlier"]]
    .hist(bins=100, color="#cccccc", ax=ax)
)

ax.axvline(marker_intensity_cutoff)

In [ ]:
sdata["segmentation_features"].obs["positive"] = (
    sdata["segmentation_features"].obs["0_intensity__area_normalized"]
    > marker_intensity_cutoff
) & (~sdata["segmentation_features"].obs["segmentation_outlier"])

### Visualize

In [ ]:
(
    sdata
    # .pl.render_images("image")
    .pl.render_labels("segmentation_mask", color="positive").pl.show(
        title="Positive Classification"
    )
)

## Shape Export

### Generate shape vectors

This step is automatically done by harpy. If other segmentation packages are used, this step can be used to vectorize the masks to vectorized shapes. 

In [ ]:
# harpy.sh.vectorize(sdata, labels_layer="segmentation_mask", output_layer="segmentation_boundaries")

### Select marker-positive cells

**Critical step**

In [ ]:
positive_cell_ids = (
    sdata["segmentation_features"]
    .obs.loc[sdata["segmentation_features"].obs["positive"], "cell_ID"]
    .to_numpy()
)

In [ ]:
positive_shapes = sdata["segmentation_boundaries"].loc[positive_cell_ids]

### Process shapes

#### Simplify

In [ ]:
positive_shapes_simplified = positive_shapes.copy()
positive_shapes_simplified.geometry = positive_shapes_simplified.geometry.simplify(
    tolerance=1
)

#### Buffer

In [ ]:
IMAGE_RESOLUTION = 0.1725

In [ ]:
positive_shapes_simplified_buffered = positive_shapes_simplified.copy()
positive_shapes_simplified_buffered.geometry = (
    positive_shapes_simplified_buffered.geometry.buffer(distance=0.2 / IMAGE_RESOLUTION)
)

### Select shapes for dissection
Strategies: 
- Randomly sample until area is reached
- Subset to specific areas

In [ ]:
selected_shapes = random_sample_by_area(gdf=positive_shapes, target_area=40_000)
selected_shapes["well"] = "B2"

### Select calibration points


**Critical step**

In [ ]:
session = Interactive(sdata)
session.run()

In [ ]:
calibration_points = sdata["calibration_points"].compute().to_numpy()

### Save shapes with py-lmd

In [ ]:
collection = pylmd.Collection(
    calibration_points=calibration_points, orientation_transform=np.eye(3), scale=1
)
collection.load_geopandas(selected_shapes, well_column="well")

collection.save(results_path / "shapes.xml")